# Part 4: Generative AI Follow-Up Notes Analysis & Recommendation

**Project:** CareLoop Clinics: No-Show Guard  **Objective:** Process unstructured call-centre follow-up notes using the Gemini Flash API, validate the extracted insights with strict Pydantic v2 schemas, evaluate prompt robustness (naive vs CRAFT), and combine quantitative risk scores with qualitative root causes into an actionable management recommendation.

### Ethical & Security Guidelines:
1. **Zero Credential Exposure:** Never hardcode or commit API keys. The key is entered interactively at runtime via `input()`.
2. **Strict Quota Management:** Google AI Studio free-tier has a limit of ~5 requests/min. Calls are throttled with `time.sleep(13)` to guarantee compliance with free-tier quotas.
3. **Zero Token Waste:** Blank/whitespace notes are detected locally and bypassed (`skipped_blank`), consuming zero API calls.

## Environment Setup & Dependency Installation

In [1]:
# In Google Colab, uncomment and run:
# !pip install -q google-genai pydantic

import os
import time
import json
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from google import genai
from google.genai import types

print("Required libraries imported successfully.")

Required libraries imported successfully.


## Task 4.1: One Validated API Call with CRAFT Prompt

- Connect to Google AI Studio using Gemini 2.5 Flash.
- Construct CRAFT prompt:
  - **Context:** CareLoop outpatient healthcare network in India.
  - **Role:** Clinical operations analyst specializing in appointment scheduling.
  - **Action:** Classify patient reason into exactly one allowed category, assign confidence (0.0 to 1.0), summarize in <= 25 words, and propose actionable operational recommendation.
  - **Tone:** Objective, professional, clinical.
  - **Format:** JSON adhering to `SCHEMA`.
- Define JSON response schema with `root_cause`, `confidence`, `summary`, `recommended_action`.
- Test on note `N001`.

In [2]:
ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"]
}

SYSTEM_TEXT = """You are an expert clinical operations analyst for CareLoop Clinics, an outpatient network in India.
Analyze the call-center note from a follow-up conversation with a patient who missed their clinic appointment.
Classify the reason into EXACTLY ONE of the following allowed root causes:
- forgot_or_no_reminder
- distance_or_transport
- cost_or_fee
- illness_or_emergency
- work_or_schedule
- other (use this when the reason is ambiguous, unclear, or not listed)

Provide:
1. root_cause: one of the exact strings above.
2. confidence: float between 0.0 and 1.0.
3. summary: concise factual summary of at most 25 words.
4. recommended_action: actionable clinical operations recommendation.
Output valid JSON conforming strictly to the requested schema."""

# Enter API Key securely at runtime
API_KEY = input("Paste your Google AI Studio API key: ")
client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-2.5-flash"

def analyse_note(note_text, model_name=MODEL_NAME):
    prompt = f"Patient Follow-Up Note:\n\"{note_text}\"\n\nClassify the root cause and provide structured recommendations."
    config_args = {
        "system_instruction": SYSTEM_TEXT,
        "temperature": 0,
        "max_output_tokens": 1000,
        "response_mime_type": "application/json",
        "response_schema": SCHEMA
    }
    try:
        config_args["thinking_config"] = types.ThinkingConfig(thinking_level="minimal")
        config = types.GenerateContentConfig(**config_args)
        response = client.models.generate_content(model=model_name, contents=prompt, config=config)
    except Exception:
        config_args.pop("thinking_config", None)
        config = types.GenerateContentConfig(**config_args)
        response = client.models.generate_content(model=model_name, contents=prompt, config=config)
    return response.text

# Test on note N001
notes_raw_df = pd.read_csv(os.path.join("data", "followup_notes.csv"))
first_note = notes_raw_df.iloc[0]['note_text']
print(f"Testing Note N001: '{first_note}'")
raw_reply_n001 = analyse_note(first_note)
data_n001 = json.loads(raw_reply_n001)
print("\nParsed Dictionary for N001:")
print(json.dumps(data_n001, indent=2))

Testing Note N001: 'Patient stated they completely forgot about the visit because no reminder message was received.'

Parsed Dictionary for N001:
{
  "root_cause": "forgot_or_no_reminder",
  "confidence": 0.95,
  "summary": "Patient stated they forgot about visit because no reminder message was received.",
  "recommended_action": "Ensure automated SMS/WhatsApp reminders are dispatched 24h prior to visit."
}


## Task 4.2: Validate and Process the 10 Notes

- Define Pydantic v2 `FollowupInsight(BaseModel)` with boundary constraints.
- Enforce root cause membership in `ALLOWED_ROOT_CAUSES`.
- Pre-screen notes: Detect blank/empty notes (`N004`) locally to bypass the API (`validation_status="skipped_blank"`, `raw_response=""`).
- Apply 13-second rate-limiting pause between API calls.
- Store all raw and validated responses without mutation in `notes_analysed.csv`.

In [3]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0.0, le=1.0)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

analysed_records = []
first_10_notes = notes_raw_df.head(10).copy()

for idx, row in first_10_notes.iterrows():
    n_id = row['note_id']
    a_id = row['appointment_id']
    n_text = row['note_text']
    
    if pd.isna(n_text) or str(n_text).strip() == "":
        analysed_records.append({
            'note_id': n_id,
            'appointment_id': a_id,
            'root_cause': 'other',
            'confidence': 0.0,
            'summary': 'No note recorded',
            'recommended_action': 'Re-call patient',
            'validation_status': 'skipped_blank',
            'raw_response': ''
        })
        print(f"Processing Note {n_id}: Blank note detected -> Skipped API call (validation_status='skipped_blank').")
        continue
        
    print(f"Processing Note {n_id}: Calling API...")
    raw_reply = analyse_note(str(n_text))
    val_status = "invalid"
    r_cause, conf, summ, rec_act = "", "", "", ""
    
    try:
        parsed_dict = json.loads(raw_reply)
        insight = FollowupInsight(**parsed_dict)
        if insight.root_cause in ALLOWED_ROOT_CAUSES:
            val_status = "valid"
            r_cause = insight.root_cause
            conf = insight.confidence
            summ = insight.summary
            rec_act = insight.recommended_action
        else:
            val_status = "invalid"
            r_cause = parsed_dict.get('root_cause', '')
    except Exception:
        val_status = "invalid"
        try:
            p = json.loads(raw_reply)
            r_cause = p.get('root_cause', '')
            conf = p.get('confidence', '')
            summ = p.get('summary', '')
            rec_act = p.get('recommended_action', '')
        except Exception:
            pass
            
    analysed_records.append({
        'note_id': n_id,
        'appointment_id': a_id,
        'root_cause': r_cause,
        'confidence': conf,
        'summary': summ,
        'recommended_action': rec_act,
        'validation_status': val_status,
        'raw_response': raw_reply
    })
    time.sleep(13)

notes_analysed_df = pd.DataFrame(analysed_records)
notes_analysed_df.to_csv("notes_analysed.csv", index=False)
print("\nExported notes_analysed.csv with 10 rows and 8 columns.")
print("\nSummary Table:")
print(notes_analysed_df[['note_id', 'root_cause', 'confidence', 'validation_status']].to_string(index=False))

Processing Note N001: Calling API...
Processing Note N002: Calling API...
Processing Note N003: Calling API...
Processing Note N004: Blank note detected -> Skipped API call (validation_status='skipped_blank').
Processing Note N005: Calling API...
Processing Note N006: Calling API...
Processing Note N007: Calling API...
Processing Note N008: Calling API...
Processing Note N009: Calling API...
Processing Note N010: Calling API...

Exported notes_analysed.csv with 10 rows and 8 columns.

Summary Table:
note_id            root_cause  confidence validation_status
   N001 forgot_or_no_reminder        0.95             valid
   N002 distance_or_transport        0.92             valid
   N003           cost_or_fee        0.94             valid
   N004                 other        0.00     skipped_blank
   N005  illness_or_emergency        0.96             valid
   N006      work_or_schedule        0.91             valid
   N007 forgot_or_no_reminder        0.93             valid
   N008 distanc

## Task 4.3: Break It On Purpose & Prompt Comparison

- **Adversarial Pydantic Testing:** Deliberately inject faulty payloads (confidence = 1.7, missing required field) and verify graceful exception handling.
- **Prompt Comparison (V1 vs V2):** Test naive prompt (`"Classify this note"`) vs CRAFT prompt on Hinglish note (`N007`) and ambiguous note (`N009`).
- **Call Accounting:** Track total API calls used across Tasks 4.1–4.3 (well under the 25 free-tier quota).

In [4]:
# (a) Pydantic Fault Injection
print("=== Part (a): Deliberate Pydantic Validation Error Trapping ===")
bad_conf = {"root_cause": "cost_or_fee", "confidence": 1.7, "summary": "Patient could not afford medicine", "recommended_action": "Offer discount"}
try:
    FollowupInsight(**bad_conf)
except ValidationError as e:
    print(f"Caught expected confidence error:\n{e}\n")

bad_missing = {"root_cause": "forgot_or_no_reminder", "confidence": 0.9, "summary": "Patient forgot the visit"}
try:
    FollowupInsight(**bad_missing)
except ValidationError as e:
    print(f"Caught expected missing field error:\n{e}\n")

# (b) V1 vs V2 Comparison on N007 & N009
print("=== Part (b): V1 (Naive) vs V2 (CRAFT) Prompt Comparison ===")
test_note_ids = ['N007', 'N009']
comp_records = []
for nid in test_note_ids:
    txt = notes_raw_df[notes_raw_df['note_id'] == nid]['note_text'].values[0]
    # Naive V1 call
    p_v1 = f"Classify this note: {txt}"
    cfg_v1 = types.GenerateContentConfig(temperature=0, response_mime_type="application/json", response_schema=SCHEMA)
    resp_v1 = client.models.generate_content(model=MODEL_NAME, contents=p_v1, config=cfg_v1)
    rc_v1 = json.loads(resp_v1.text).get('root_cause', '')
    time.sleep(13)
    
    # V2 from Task 4.2
    rc_v2 = notes_analysed_df[notes_analysed_df['note_id'] == nid]['root_cause'].values[0]
    comp_records.append({
        'note_id': nid,
        'note_text': txt,
        'V1_naive_root_cause': rc_v1,
        'V1_in_allowed': rc_v1 in ALLOWED_ROOT_CAUSES,
        'V2_craft_root_cause': rc_v2,
        'V2_in_allowed': rc_v2 in ALLOWED_ROOT_CAUSES
    })

print(pd.DataFrame(comp_records).to_string(index=False))
print("\nDecision: Deploy V2 (CRAFT Prompt). While V1 works on straightforward Hinglish phrases, it hallucinates unconstrained categories (e.g., 'personal_reasons') on ambiguous inputs. V2 constrains outputs strictly to allowed operational domains and falls back to 'other'.")
print("\n=== Part (c): API Call Accounting ===")
print("Total API calls executed: 1 (Task 4.1) + 9 (Task 4.2) + 2 (Task 4.3 V1 tests) = 12 calls (Budget ceiling: 25).")

=== Part (a): Deliberate Pydantic Validation Error Trapping ===
Caught expected confidence error:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]

Caught expected missing field error:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'forgot_or_no_reminder', 'confidence': 0.9, 'summary': 'Patient forgot the visit'}, input_type=dict]

=== Part (b): V1 (Naive) vs V2 (CRAFT) Prompt Comparison ===
note_id                                                                                note_text   V1_naive_root_cause  V1_in_allowed   V2_craft_root_cause  V2_in_allowed
   N007  Patient bola ki reminder nahi mila tha so forgot ho gaya date, agle hafte book karenge. forgot_or_no_reminder           True forgot_or_no_reminder           True
   N009 Patient was vague and cited personal private reasons without specifying any clear is

## Task 4.4: Strategic Recommendation & Cross-Tabulation

- Merge qualitative call-center analysis (`notes_analysed.csv`) with quantitative booking data (`appointments_clean.csv`) on `appointment_id`.
- Cross-tabulate `root_cause` against `reminder_sent` to identify systematic service gaps.
- Synthesize evidence-based management recommendations.

In [5]:
df_clean = pd.read_csv('appointments_clean.csv')
merged_df = pd.merge(notes_analysed_df, df_clean, on='appointment_id', how='inner')
print(f"Merged dataset size: {len(merged_df)} rows")

ct = pd.crosstab(merged_df['root_cause'], merged_df['reminder_sent'], margins=True)
print("\n=== Cross-Tabulation: Root Cause vs Reminder Sent ===")
print(ct)

print("\nKey Empirical Insight:")
print("- Exactly 2 of 10 sampled no-show notes cited 'forgot_or_no_reminder' (N001, N007).")
print("- Both (100%) of these patients were from the unreminded group (reminder_sent = 'No').")
print("- Exactly 5 of the 10 sampled no-show patients were unreminded, highlighting an immediate operational fix.")

Merged dataset size: 10 rows

=== Cross-Tabulation: Root Cause vs Reminder Sent ===
reminder_sent          No  Yes  All
root_cause                         
cost_or_fee             0    1    1
distance_or_transport   1    1    2
forgot_or_no_reminder   2    0    2
illness_or_emergency    0    2    2
other                   1    1    2
work_or_schedule        1    0    1
All                     5    5   10

Key Empirical Insight:
- Exactly 2 of 10 sampled no-show notes cited 'forgot_or_no_reminder' (N001, N007).
- Both (100%) of these patients were from the unreminded group (reminder_sent = 'No').
- Exactly 5 of the 10 sampled no-show patients were unreminded, highlighting an immediate operational fix.
